# MobApp Studio, workshop 3 — The report is the deliverable

The project asks for two files. One is the notebook. The other is `link_to_blog_post.txt`: a
**published** post, written — in the brief's own words — "as a *scientific experiment*", with your
assumptions, every step, and the conclusion, ending on what you would investigate next.

That second file is half the submission, and it is the half people improvise the night before.
It is also the only part John's VPs will actually read.

A scientific report is not a prettier notebook. It is a different structure:

| section | the question it answers |
|---|---|
| **Question** | what decision does this inform? |
| **Assumptions** | what did I have to take on faith? |
| **Method** | what did I do, precisely enough to repeat? |
| **Results** | what did the data say? |
| **Limitations** | when is this answer wrong? |
| **Next** | what would I measure to find out? |

Today you write each one, backed by a number you compute here.

In [ ]:
# ABOUTME: MobApp Studio workshop 3 - turning the analysis into a defensible written report.
# ABOUTME: Assumptions with numbers, a claims-to-evidence table, honest limitations.

from pathlib import Path
import numpy as np
import pandas as pd

def qwasar_file(name):
    """Locate a dataset fetched by workshops/get_data.py, from wherever this notebook was opened."""
    for up in range(4):
        p = Path(*([".."] * up), "data", name)
        if p.exists():
            return p
    raise FileNotFoundError(
        f"{name} is missing. Run `uv run get_data.py` in the workshops/ folder first - "
        "these notebooks only ever use the real dataset the project links to.")

# The Google Play export the project points at. 10,841 rows, exactly as published.
raw = pd.read_csv(qwasar_file("googleplaystore.csv"))

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 20)
print(f"loaded {len(raw):,} rows x {raw.shape[1]} columns from the real Google Play export")


In [ ]:
# The cleaning from workshop 1, so this notebook stands alone.
df = raw[raw.Category.astype(str).str.match(r"^[A-Z_]+$")].copy()
df["Reviews"] = pd.to_numeric(df.Reviews, errors="coerce")
df = df.sort_values("Reviews", ascending=False).drop_duplicates(subset="App").reset_index(drop=True)
df["Installs"] = pd.to_numeric(df.Installs.astype(str).str.replace(r"[,+]", "", regex=True), errors="coerce")
df["Price"] = pd.to_numeric(df.Price.astype(str).str.replace("$", "", regex=False), errors="coerce")
df["Paid"] = df.Price > 0
rows_dropped = len(raw) - len(df)
print(f"{len(raw):,} rows in -> {len(df):,} apps out ({rows_dropped:,} removed)")

## 1. An assumption is a number you cannot verify

John asked for the market size **in dollars**. The data has prices and installs, so the arithmetic
is obvious — and the result is not a measurement, it is a model with at least four assumptions
baked in.

Watch the honest version of that calculation.

In [ ]:
paid = df[df.Paid]
revenue_proxy = float((paid.Price * paid.Installs).sum())
print(f"paid-app revenue proxy: ${revenue_proxy:,.0f}")
print(f"  from {len(paid):,} paid apps out of {len(df):,}")
print(f"  free apps are {(~df.Paid).mean():.1%} of the store and "
      f"{df.loc[~df.Paid, 'Installs'].sum() / df.Installs.sum():.2%} of all installs")

Every one of these is an assumption that could be wrong:

1. **Installs x price = revenue.** It ignores refunds, regional pricing, and the fact that Google
   keeps a cut (30 % historically, 15 % on the first million).
2. **Install counts are exact.** They are not — they are buckets, so this is a *lower bound*.
3. **Paid apps are where the money is.** 99.9 % of installs are free apps monetising through ads
   and in-app purchases, none of which is in this dataset. The real market is far larger than this
   number and this data cannot see it.
4. **A download is a purchase.** Pre-installed and re-installed apps count too.

Stating these does not weaken the report. It is the difference between "the market is $291 M" —
which is wrong — and "paid-app gross sales are **at least** ~$291 M; this excludes ads and in-app
purchases, which is where most revenue actually sits" — which is defensible and more useful.

**Rule of thumb:** if a number required a step the data cannot justify, that step is an assumption.
Write it down the moment you take it; you will not remember it on Sunday night.

### Exercise 1 — write the assumptions for your own headline

In [ ]:
# TODO: write 3 or more assumptions behind the "$ size of the market" figure.
#       One sentence each, each naming what could make the number wrong.
assumptions = [
    "",
]
for i, a in enumerate(assumptions, 1):
    print(f"{i}. {a}")

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("at least three assumptions", lambda: len([a for a in assumptions if a.strip()]) >= 3)
check("each is a real sentence", lambda: all(len(a.split()) >= 8 for a in assumptions if a.strip()))

## 2. Every claim needs a number, and every number needs a line of code

The failure mode of a report like this is the confident sentence nobody can trace: *"Games dominate
the store."* Dominate how? Measured on what? Compared to what?

Build the table that stops that happening: claim, evidence, where the evidence came from.

### Exercise 2 — the claims-to-evidence table

Compute the four headline numbers your report will quote, and round each to the precision the
data actually supports.

In [ ]:
# TODO: fill in each number from `df`.
total_installs = None            # all installs, after cleaning
top_category = None              # the category with the most installs (a string)
top_category_share = None        # its share of all installs, in percent
best_ratio_category = None       # best installs-per-app, among categories with >= 50 apps

claims = {
    "the store is big": f"{total_installs} installs across {len(df):,} apps",
    "one category leads": f"{top_category} takes {top_category_share}% of installs",
    "but crowded": f"best installs-per-app is {best_ratio_category}",
}
for k, v in claims.items():
    print(f"- {k}: {v}")

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("all four numbers computed",
      lambda: None not in (total_installs, top_category, top_category_share, best_ratio_category))
check("the leading category is the one with the most installs",
      lambda: top_category == df.groupby("Category").Installs.sum().idxmax())
check("the ratio winner is a different category", lambda: best_ratio_category != top_category)
check("no false precision in the claims", lambda: "75,322,526,427" not in " ".join(claims.values()))

That last check is the one students argue about. `75,322,526,427` looks more rigorous than
"roughly 75 billion". It is the opposite: it claims eleven digits of accuracy from data that was
rounded into twenty buckets before you ever saw it.

**Rule of thumb:** precision is a claim. Only make it when you can back it.

## 3. The limitations section is where you get your marks

Every analysis has limits. The difference between a student report and a professional one is not
that the professional has fewer — it is that the professional **found them first**.

Yours are not hypothetical. You measured them in the last two workshops.

### Exercise 3 — write the limitations from your own evidence

For each known weakness, compute the number that quantifies it, then write the sentence.

In [ ]:
# TODO: compute each number, then write the sentence that would appear in your blog post.
n_dropped = None             # rows removed during cleaning
pct_no_rating = None         # share of apps with no rating, in percent
n_install_levels = None      # how many distinct install values exist
snapshot_year = None         # the year the data describes

limitations = []             # one sentence per limitation, each containing its number
for line in limitations:
    print("-", line)

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("four numbers computed",
      lambda: None not in (n_dropped, pct_no_rating, n_install_levels, snapshot_year))
check("at least four limitations", lambda: len(limitations) >= 4)
check("each one carries a number", lambda: all(any(ch.isdigit() for ch in l) for l in limitations))

Read those back. None of them says "the data was messy" — each names a specific weakness, with a
number, and tells the reader when your conclusion stops being true. That is what the brief means
by a scientific report.

**The punchline:** a limitation you state is a sign of rigour. The same limitation found by your
reviewer is a hole in your work.

## 4. The opening to what comes next

The brief ends the structure with *"an opening to what you follow, what to experiment next?"*.
A good next step is **specific, feasible, and would change a decision**. "More data" is none of
those.

### Exercise 4 — propose the next experiment

In [ ]:
# TODO: propose 2 next experiments. For each: what you would measure, what data you would need,
#       and which decision the answer would change.
next_experiments = [
    {"measure": "", "data_needed": "", "decision_it_changes": ""},
]
for e in next_experiments:
    print(f"- measure: {e['measure']}\n  needs: {e['data_needed']}\n  changes: {e['decision_it_changes']}\n")

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("two experiments", lambda: len(next_experiments) >= 2)
check("each is filled in", lambda: all(all(v.strip() for v in e.values()) for e in next_experiments))
check("each names a decision", lambda: all(len(e["decision_it_changes"].split()) >= 6 for e in next_experiments))

## 5. Before you publish

The post goes on any platform you like — Medium, dev.to, a GitHub Pages blog, your own site — and
its URL goes in `link_to_blog_post.txt`. Check it against this list first:

- [ ] A reader who has never seen the dataset understands the question in the first paragraph.
- [ ] Every chart has a caption saying what to notice, not just what it is.
- [ ] Every number in the text can be traced to a cell in the notebook.
- [ ] Anything excluded is stated, with a count and a reason.
- [ ] Precision matches the data ("roughly 75 billion", not eleven digits).
- [ ] The limitations section exists and is specific.
- [ ] The conclusion answers John's actual question: which category, and why.
- [ ] There is a next experiment, and it would change a decision.
- [ ] The notebook runs top to bottom on a fresh kernel.
- [ ] The link in `link_to_blog_post.txt` opens without a login.

That last one fails more submissions than anything on the list. Open your link in a private
browser window before you submit.

## What to take away

1. **The report is half the project**, and it is graded on structure, not prose style.
2. **An assumption is any step the data cannot justify.** Write it down when you take it.
3. **Every claim gets a number; every number gets a line of code.** Untraceable sentences are
   where reports fall apart under questions.
4. **Round to the precision you have.** Bucketed data does not support eleven digits.
5. **Find your own limitations.** The ones you name are rigour; the ones your reviewer names are
   holes.
6. **A good next experiment is specific, feasible, and decision-changing.**

### Take it home
Write the post. Then hand the draft to someone who has not seen the data and ask them one question:
*"what should the company do, and why?"* If they cannot answer from the post alone, the conclusion
is buried — and that is a ten-minute fix now, rather than a question you cannot answer in the review.